In [5]:
from utils.data_prep import get_clean_combined_data
from models.train_models import train_evaluate_model

In [2]:
model_b_all_nopca_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": True,
    "conflict_only": False,
    "k": 1.75,
    "event_col": "event_type",
    "n_splits": 5,
    "use_pca": False,
    "price_recency": True,
}
model_b_all_nopca_xgb_params = {
    "max_depth": 5,
    "min_child_weight": 1,
    "max_delta_step": 1,
    "gamma": 5,
    "learning_rate": 0.01,
    "subsample": 0.6,
    "colsample_bytree": 0.8,
    "reg_alpha": 1.0,
    "reg_lambda": 5,
    "colsample_bylevel": 1.0,
}

In [1]:
model_a_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": False,
    "conflict_only": None,
    "k": 1.75,
    "event_col": "sub_event_type",
    "n_splits": 5,
    "use_pca": False,
    "price_recency": True,
}
model_a_xgb_params = {
    "max_depth": 3,
    "min_child_weight": 1,
    "max_delta_step": 0,
    "gamma": 0,
    "learning_rate": 0.01,
    "subsample": 0.6,
    "colsample_bytree": 0.8,
    "reg_alpha": 2.0,
    "reg_lambda": 1,
    "colsample_bylevel": 1.0,
}

In [2]:
config = model_a_config
params = model_a_xgb_params

In [ ]:
config = model_b_all_nopca_config
params = model_b_all_nopca_xgb_params

In [3]:
data_sources = [
    src
    for src, include in zip(
        ["food", "rain", "text"],
        [config["include_food"], config["include_rain"], config["include_text"]],
    )
    if include
]

In [6]:
model_data, predictor_cols = get_clean_combined_data(
    data_sources=data_sources,
    k=config["k"],
    event_col=config["event_col"],
    conflict_only_embeddings=config["conflict_only"],
    price_recency=config["price_recency"],
)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:hdx.api.configuration:No HDX base configuration parameter. Using default base configuration file: /Users/evie.jones/Downloads/msc-final-project/.venv/lib/python3.14/site-packages/hdx/api/hdx_base_configuration.yaml.
INFO:hdx.api.configuration:Loading HDX base configuration from: /Users/evie.jones/Downloads/msc-final-project/.venv/lib/python3.14/site-packages/hdx/api/hdx_base_configuration.yaml
INFO:hdx.api.configuration:No HDX configuration parameter and no configuration file at default path: /Users/evie.jones/.hdx_configuration.yaml.
INFO:hdx.api.configuration:Read only access to HDX: True
INFO:HDXClient:Reading local file: data/hdx/hdx_sudan_food_prices.csv
INFO:Name mapping:Renamed 

In [7]:
model_data

,region,year_month,conflict_event_count,rolling_mean_6m,rolling_std_6m,escalation_threshold,target_escalation,abduction_forced_disappearance,agreement,air_drone_strike,...,shelling_artillery_missile_attack,violent_demonstration,fatalities,price_millet,price_sorghum,price_wheat_flour,months_since_reading_millet,months_since_reading_sorghum,months_since_reading_wheat_flour,rainfall_3m_anomaly
0,Al Jazirah,2018-01,2,0.000000,0.000000,0.000000,1,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,93.058950
1,Blue Nile,2018-01,1,2.166667,1.722401,5.180869,0,0.0,0.0,0.0,...,0.0,0.0,12.0,1.125714,0.913333,NaN,0.0,0.0,NaN,96.174330
2,Central Darfur,2018-01,13,7.500000,3.674235,13.929911,0,0.0,0.0,0.0,...,0.0,0.0,3.0,1.302857,0.906667,NaN,0.0,0.0,NaN,70.966080
3,East Darfur,2018-01,1,1.166667,1.169045,3.212496,0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.480000,0.946667,NaN,0.0,0.0,NaN,81.369190
4,Gedaref,2018-01,0,0.166667,0.408248,0.881101,0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,124.561620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1723,South Darfur,2025-12,35,27.333333,9.770705,44.432066,0,5.0,0.0,6.0,...,0.0,0.0,36.0,2.391429,1.166667,6.79,0.0,0.0,0.0,105.487110
1724,South Kordofan,2025-12,64,18.333333,10.838204,37.300190,1,4.0,0.0,8.0,...,1.0,0.0,200.0,3.391429,8.750000,20.62,3.0,0.0,0.0,100.186264
1725,West Darfur,2025-12,8,3.666667,2.804758,8.574993,0,1.0,0.0,2.0,...,0.0,0.0,11.0,2.205714,1.823333,5.12,0.0,0.0,0.0,89.121414
1726,West Kordofan,2025-12,42,43.000000,16.994117,72.739704,0,5.0,0.0,26.0,...,1.0,0.0,256.0,2.462857,1.743333,6.33,0.0,0.0,0.0,99.838135


In [ ]:
final_params = {
    **params,
    "k": config["k"],
    "event_col": config["event_col"],
    "n_splits": config["n_splits"],
    "use_pca": config["use_pca"],
}

results, best_params, shap_importance, onset_predictions = train_evaluate_model(
    model_data,
    predictor_cols,
    final_params,
    best_params=True,  # skip RandomizedSearchCV
    use_pca=config["use_pca"],
    compute_shap=True,
    shap_sample_size=2000,
    return_onset_predictions=True,
)

INFO:Cross validation:Cross-validation testing splits:
